# 02 · Механизмы современного трекинга и честная оценка MOT
К лекции 8. Девять TODO: confidence, две стадии с допустимостью low, CLEAR-счетчики,
глобальная ID-оценка, HOTA-alpha, робастная оценка движения камеры, матрицы аудита,
полный переход трекера через кадр и собственные интеграционные контрпримеры.

Основной эксперимент использует точную синтетическую GT и одинаковые входные detections. Полные MOTA, IDF1, HOTA считает официальный TrackEval. В конце — отдельный качественный разбор реального видео без GT.

Работа рассчитана на несколько этапов: демонстрация и ручные расчёты, реализации
и тесты, эксперимент и защита. Планируйте 6–9 часов самостоятельной работы вместе
с демо и отчётом; это ориентир, а не измеренное время студентов. Вычисления на CPU
занимают секунды/минуты. Нужны numpy, scipy, matplotlib, opencv-python, доступ к
GitHub при первом заполнении временного кэша. TrackEval source и видео проверяются
по SHA. Нет обучения CNN и больших весов. Все функции содержатся в notebook.

**Граница реализации:** BYTE-подобная схема использует скорость рамки, high для
active/lost и low только для ранее active. Нет YOLOX, Kalman, unconfirmed и отдельного
порога рождения. Границы confidence заданы явно (score ≥ high), а `match_boxes`
максимизирует число допустимых пар, затем IoU; это отличается от assignment авторов.
Компенсация камеры — только трансляция по заданным точкам фона. Matching официальной
HOTA не заменяется вашим агрегатором.

In [ ]:
# Запускайте первой. В Colab сохраняем штатные PyTorch/CUDA и ipywidgets.
import importlib.util, subprocess, sys
if sys.version_info < (3, 10):
    raise RuntimeError('Нужен Python 3.10 или новее.')
_packages = {'numpy': 'numpy>=1.26,<3', 'matplotlib': 'matplotlib>=3.8,<4', 'scipy': 'scipy>=1.11,<2', 'cv2': 'opencv-python-headless==4.12.0.88'}
_missing = [spec for module, spec in _packages.items()
            if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])
    raise RuntimeError('Пакеты установлены. Перезапустите среду выполнения и выполните ноутбук сверху.')
import numpy, matplotlib
if not (1 <= int(numpy.__version__.split('.')[0]) < 3):
    raise RuntimeError('Нужен NumPy 1.x или 2.x; используйте отдельное окружение.')
if int(matplotlib.__version__.split('.')[0]) != 3:
    raise RuntimeError('Нужен Matplotlib 3.x; используйте отдельное окружение.')
_versions = {'python': sys.version.split()[0], 'numpy': numpy.__version__,
             'matplotlib': matplotlib.__version__}
if 'scipy' in _packages:
    import scipy
    if int(scipy.__version__.split('.')[0]) != 1:
        raise RuntimeError('Нужен SciPy 1.x; используйте отдельное окружение.')
    _versions['scipy'] = scipy.__version__
if 'cv2' in _packages:
    import cv2
    _versions['opencv'] = cv2.__version__
    _required_features = ()
    _absent = [name for name in _required_features if not hasattr(cv2, name)]
    if _absent:
        raise RuntimeError('В OpenCV нет ' + ', '.join(_absent) +
                           '. Установите один вариант OpenCV 4.12.0.88 в отдельной среде и перезапустите ядро.')
if 'ipywidgets' in _packages:
    import ipywidgets
    _versions['ipywidgets'] = ipywidgets.__version__
    if int(ipywidgets.__version__.split('.')[0]) not in (7, 8):
        raise RuntimeError('Нужен ipywidgets 7.x или 8.x; в Colab 2026.07 оставьте штатный 7.7.1.')
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
print('Окружение:', _versions)
print('Вычисления этой лабораторной: CPU; GPU не требуется.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy.optimize import linear_sum_assignment
from pathlib import Path
import time,json,csv,os,hashlib
plt.rcParams.update({'figure.dpi':110,'font.size':10})

In [ ]:
# Official source files, pinned revision. License is downloaded alongside code.
import urllib.request,tempfile,types,sys,importlib
TRACKEVAL_REV='12c8791b303e0a0b50f753af204249e622d0281a'
TRACKEVAL_HASHES={'LICENSE': 'f7e548c5729e464d878dd4abf691e323bbea028a756d08395d88b83e43dd2d86', 'trackeval/metrics/hota.py': 'c582255c3d36bdb49c3ceeb2a5bd912cee009190dfcfe465b4173d03aaa97d6a', 'trackeval/metrics/clear.py': 'ab3963eac401073fe1b2957c734af1e531368f42fca49627c14422350b2744f0', 'trackeval/metrics/identity.py': 'e76b1bdcbf5b193662431596a984cf70a5be32e903eb5f2a3d9921b4762f93ba', 'trackeval/metrics/_base_metric.py': 'dac8b87bf3801269aa280b1cf91184f483de4f4e5c158bd07a4b1733b1a5722b', 'trackeval/_timing.py': '62e60638efc675bccc35c7da65158f0f19ce31f332121efe7cc748bd92ad2b30', 'trackeval/utils.py': 'c082ecfbb5061619642b82a034454022806f8e870f774e54dbfbdacaaf097132'}
cache=Path(tempfile.gettempdir())/'applied-ai-trackeval'/TRACKEVAL_REV
root=Path(os.environ.get('COURSE_TRACKEVAL_PATH',cache))
for relative,expected in TRACKEVAL_HASHES.items():
 target=root/relative
 if not target.exists():
  target.parent.mkdir(parents=True,exist_ok=True)
  url=f'https://raw.githubusercontent.com/JonathonLuiten/TrackEval/{TRACKEVAL_REV}/{relative}'
  with urllib.request.urlopen(url,timeout=60) as response:target.write_bytes(response.read())
 assert hashlib.sha256(target.read_bytes()).hexdigest()==expected,'TrackEval source checksum mismatch: '+relative
# Old revision uses removed NumPy builtin aliases. No metric formulas are changed.
if 'float' not in np.__dict__:np.float=float
if 'int' not in np.__dict__:np.int=int
# Load only official metric modules, without unrelated dataset adapters.
namespace='_course_trackeval_pinned'
for name,path in [(namespace,root/'trackeval'),(namespace+'.metrics',root/'trackeval/metrics')]:
 module=types.ModuleType(name);module.__path__=[str(path)];module.__package__=name;sys.modules[name]=module
HOTA=importlib.import_module(namespace+'.metrics.hota').HOTA
CLEAR=importlib.import_module(namespace+'.metrics.clear').CLEAR
Identity=importlib.import_module(namespace+'.metrics.identity').Identity
METRICS={'clear':CLEAR({'PRINT_CONFIG':False,'THRESHOLD':.5}),'identity':Identity({'PRINT_CONFIG':False,'THRESHOLD':.5}),'hota':HOTA()}
print('Official TrackEval revision verified:',TRACKEVAL_REV)

In [ ]:
def iou_matrix(a,b):
 a=np.asarray(a,float).reshape(-1,4);b=np.asarray(b,float).reshape(-1,4)
 inter=np.maximum(0,np.minimum(a[:,None,2:],b[None,:,2:])-np.maximum(a[:,None,:2],b[None,:,:2])).prod(2)
 union=(a[:,2:]-a[:,:2]).prod(1)[:,None]+(b[:,2:]-b[:,:2]).prod(1)[None,:]-inter
 return np.divide(inter,union,out=np.zeros_like(inter),where=union>0)

def match_boxes(track_boxes,det_boxes,threshold):
 sim=iou_matrix(track_boxes,det_boxes);n,m=sim.shape
 if not n or not m:return []
 # Lexicographic maximum cardinality among allowed pairs, then maximum IoU.
 # A bonus > maximal total IoU makes this objective explicit.
 score=np.zeros((n+m,n+m));score[:n,:m]=np.where(sim>=threshold,n+m+1+sim,0)
 rows,cols=linear_sum_assignment(-score)
 return [(int(i),int(j)) for i,j in zip(rows,cols) if i<n and j<m and sim[i,j]>=threshold]

def make_metric_data(gt_frames,pred_frames):
 assert len(gt_frames)==len(pred_frames)
 gids=sorted({int(i) for f in gt_frames for i in f['ids']});pids=sorted({int(i) for f in pred_frames for i in f['ids']})
 gm={x:i for i,x in enumerate(gids)};pm={x:i for i,x in enumerate(pids)}
 data={'num_gt_ids':len(gids),'num_tracker_ids':len(pids),'num_gt_dets':sum(len(f['ids']) for f in gt_frames),'num_tracker_dets':sum(len(f['ids']) for f in pred_frames),'num_timesteps':len(gt_frames),'gt_ids':[],'tracker_ids':[],'similarity_scores':[]}
 for g,p in zip(gt_frames,pred_frames):
  assert len(set(g['ids']))==len(g['ids']) and len(set(p['ids']))==len(p['ids'])
  data['gt_ids'].append(np.array([gm[int(i)] for i in g['ids']],int));data['tracker_ids'].append(np.array([pm[int(i)] for i in p['ids']],int))
  data['similarity_scores'].append(iou_matrix(g['boxes'],p['boxes']))
 return data

def metric_row(results):
 c,i,h=results['clear'],results['identity'],results['hota']
 return {**{k:float(c[k]) for k in ['MOTA','CLR_TP','CLR_FP','CLR_FN','IDSW']},**{k:float(i[k]) for k in ['IDF1','IDTP','IDFP','IDFN']},**{k:float(np.mean(h[k])) for k in ['HOTA','DetA','AssA']}}

def official(data):return {name:metric.eval_sequence(data) for name,metric in METRICS.items()}

def simple_case(ids,fp_per_frame=0,overlap=1.):
 # One GT per frame. -1 means missing detection. Extra predictions never overlap.
 gt=[];pred=[]
 for t,tid in enumerate(ids):
  gt.append({'ids':[0],'boxes':[[0,0,10,10]]});ps=[];boxes=[]
  if tid>=0:
   dx=10*(1-overlap)/(1+overlap);ps.append(int(tid));boxes.append([dx,0,10+dx,10])
  for j in range(fp_per_frame):ps.append(100+j);boxes.append([30+15*j,0,40+15*j,10])
  pred.append({'ids':ps,'boxes':boxes})
 return make_metric_data(gt,pred)

## Гипотеза и протокол до запуска
**Заполните:** когда слабые детекции сохранят ID, когда они окажутся фоном, почему разрешение low births может ухудшить FP, какие измерения опровергнут гипотезу.

Val: seeds 101–104; test: 201–204; по 40 кадров с двумя пересекающимися объектами. Входные boxes/scores одинаковы у всех вариантов. High = .6, IoU gate = .2, max_age = 2, динамика и генератор фиксированы. Меняем только low = .1/.25/.4, выбираем наибольшую **официальную pooled val HOTA**; ties → больший low. Выбор записывается до создания test.

Сравнения на test объявлены заранее: high-only baseline; выбранная двухстадийная схема; абляция той же схемы с разрешенными low births. Последняя меняет только правило рождения. Полные метрики объединяются официальным `combine_sequences`, затем HOTA усредняется по alpha. Не усредняйте проценты отдельных видео вместо официального объединения.

## TODO 1 · Разбиение confidence
Проверьте обе границы и пустой массив. Ни одна detection не должна одновременно оказаться high и low.

In [ ]:
def confidence_groups(scores,low,high):
    """Finite 1D scores in [0,1], 0<=low<high<=1; otherwise ValueError.
    Return integer index arrays high(score>=high), low(low<=score<high), ignored.
    Empty input is valid; groups are disjoint."""
    raise NotImplementedError('TODO 1: confidence groups')

In [ ]:
hi,lo,ig=confidence_groups([.6,.2,.199,1],.2,.6)
assert hi.tolist()==[0,3] and lo.tolist()==[1] and ig.tolist()==[2]
assert all(len(x)==0 for x in confidence_groups([],.2,.6))
for bad in [([np.nan],.2,.6),([.5],.6,.6),([1.1],.2,.6)]:
 try:confidence_groups(*bad)
 except ValueError:pass
 else:raise AssertionError('invalid confidence')
print('Partition tests passed')

## TODO 2 · Высокие scores первыми
Используйте готовый общий match_boxes. Сохраните исходные индексы tracks/detections между стадиями; low не создает рождения. При пустой сцене допустимы рождения только из high.
В LOW допускаются только строки low_eligible; HIGH использует все tracks. Проверьте lost→low отдельно от lost→high.

In [ ]:
def two_stage(track_boxes,det_boxes,scores,low,high,gate,use_low=True,low_eligible=None):
    """Use supplied match_boxes. High matches first, low only to remaining tracks.
    Return pairs in original indices, unmatched track indices, births from
    unmatched HIGH detections only. Low/ignored never create births.
    Empty arrays have shape (0,4); use_low=False gives high-only baseline.
    low_eligible: bool mask of tracks allowed into LOW (None=all). Old lost
    tracks may match HIGH but not LOW. Invalid mask shape raises ValueError."""
    raise NotImplementedError('TODO 2: two-stage association')

In [ ]:
tracks=np.array([[0,0,10,10],[20,0,30,10]])
det=np.array([[1,0,11,10],[21,0,31,10],[50,0,60,10]])
p,u,b=two_stage(tracks,det,[.9,.3,.3],.1,.6,.2)
assert set(p)=={(0,0),(1,1)} and u==[] and b==[]
p,u,b=two_stage(tracks,det,[.9,.3,.3],.1,.6,.2,False);assert p==[(0,0)] and u==[1] and b==[]
p,u,b=two_stage(np.empty((0,4)),det,[.9,.3,.3],.1,.6,.2);assert p==[] and b==[0]
p,u,b=two_stage(tracks,np.empty((0,4)),[],.1,.6,.2);assert u==[0,1] and b==[]
print('Two-stage tests passed')
p,u,b=two_stage(tracks,det,[.9,.3,.3],.1,.6,.2,True,[True,False])
assert p==[(0,0)] and u==[1] and b==[]
# High has worse IoU than low, yet high-first must win.
p,u,b=two_stage([[0,0,10,10]],[[2.5,0,12.5,10],[0,0,10,10]],[.9,.3],.1,.6,.2)
assert p==[(0,0)]

## TODO 3 · Счетчики по известным совпадениям
Matching уже дан. Храните последний matched predicted ID каждого GT между кадрами. Это проверяет формулу и счетчики, но не воспроизводит полный CLEAR matching.

In [ ]:
def clear_counts(events):
    """Each event: gt_ids, pred_ids, matches as (gt_id,pred_id).
    Validate membership/unique IDs/one-to-one pairs. Accumulate TP,FP,FN,N_GT.
    IDSW: changed predicted ID relative to last matched occurrence of same GT,
    even across missing frames. Return counts and MOTA; N_GT0 -> MOTA NaN.
    Matches are GIVEN; this is not the full CLEAR matching algorithm."""
    raise NotImplementedError('TODO 3: CLEAR counts from known matches')

In [ ]:
events=[{'gt_ids':[0],'pred_ids':[i],'matches':[(0,i)]} for i in [0,0,1,1]]
r=clear_counts(events);assert r['MOTA']==.75 and r['IDSW']==1 and r['TP']==4
r=clear_counts([events[0],{'gt_ids':[0],'pred_ids':[],'matches':[]},events[2]])
assert r['FN']==1 and r['IDSW']==1 and np.isclose(r['MOTA'],1/3)
assert np.isnan(clear_counts([])['MOTA'])
try:clear_counts([{'gt_ids':[0],'pred_ids':[2],'matches':[(0,3)]}])
except ValueError:pass
else:raise AssertionError('nonexistent match')
print('CLEAR counter tests passed')

## TODO 4 · Глобальное соответствие траекторий
Одному GT соответствует не более одного predicted ID на всю sequence. Нельзя взять максимум отдельно для каждого кадра или каждую ненулевую ячейку C.

In [ ]:
def global_id_scores(compatibility,gt_counts,pred_counts):
    """C[g,p] counts frames where this pair has IoU>=.5; g/p counts full lengths.
    Use global one-to-one assignment maximizing total compatible detections.
    Return IDTP,IDFN,IDFP,IDF1 (both totals0 -> IDF1 0).
    Validate shapes, finite nonnegative counts and C[g,p]<=min(g_count,p_count).
    Row sums MAY exceed g_count when candidate boxes overlap."""
    raise NotImplementedError('TODO 4: global identity assignment')

In [ ]:
assert global_id_scores([[2,2]],[4],[2,2])=={'IDTP':2.,'IDFN':2.,'IDFP':2.,'IDF1':.5}
assert global_id_scores(np.empty((0,2)),[],[2,3])['IDFP']==5
assert global_id_scores(np.empty((0,0)),[],[])['IDF1']==0
# Duplicate candidate compatibility: each GT detection overlaps both predictions.
assert global_id_scores([[4,4]],[4],[4,4])['IDTP']==4
print('Identity tests passed')

## TODO 5 · HOTA при уже заданных совпадениях
TP = сумма C. Для каждой пары association Jaccard = C/(GT length + predicted length − C). AssA — среднее Jaccard по TP, поэтому каждая ячейка взвешивается C. Затем HOTA_alpha = sqrt(DetA·AssA). Это только агрегация одного порога; официальный matching и 19 alpha остаются у TrackEval.

In [ ]:
def hota_alpha_from_matches(match_counts,gt_counts,pred_counts,fn,fp):
    """C already contains ONE-TO-ONE matched counts at a fixed alpha.
    Compute DetA, weighted association Jaccard AssA, HOTA_alpha=sqrt(DetA*AssA).
    Validate shapes/counts, row<=GT lengths, column<=pred lengths,
    GT total=TP+FN and pred total=TP+FP. No TP => all scores0.
    This aggregation is NOT full HOTA matching or averaging over alpha."""
    raise NotImplementedError('TODO 5: HOTA alpha aggregation')

In [ ]:
r=hota_alpha_from_matches([[2,2]],[4],[2,2],0,0)
assert r['DetA']==1 and r['AssA']==.5 and np.isclose(r['HOTA_alpha'],np.sqrt(.5))
assert hota_alpha_from_matches(np.empty((0,0)),[],[],0,0)['HOTA_alpha']==0
try:hota_alpha_from_matches([[2]],[1],[2],0,0)
except ValueError:pass
else:raise AssertionError('invalid matching totals')
print('HOTA aggregation tests passed')

## TODO 6 · Измерить движение камеры, не используя GT
Вход — соответствующие точки **фона** в соседних кадрах. Модель здесь только
трансляционная: d_k = after_k − before_k. Сначала найдите покоординатную медиану,
затем inliers с евклидовым остатком ≤ max_residual и медиану по этим inliers.
Оценка принимается при min_points inliers и строгом большинстве всех пар;
иначе верните нулевой сдвиг и valid=False. Верните также маску inliers.

Проверьте выбросы, отсутствие точек и недостаточную поддержку. Это учебный
частный случай GMC: в BoT-SORT преобразование оценивается по изображениям и
затрагивает Kalman mean/covariance. Здесь нет поворота, масштаба и Kalman.
Объясните, почему медиана движения самих людей не является оценкой фона.

In [ ]:
def estimate_camera_translation(before, after, min_points=3, max_residual=1.):
    """Return (shift[2], inlier_mask[N], valid). Same finite Nx2 arrays.
    Reject bad shapes/nonfinite/invalid limits with ValueError; empty is valid.
    Initial coordinate median -> residual gate -> median of inliers.
    Accept only >=min_points AND >N/2 inliers, else shift=zeros(2)."""
    raise NotImplementedError('TODO 6: robust background translation')

In [ ]:
bg=np.array([[0,0],[10,0],[0,10],[10,10],[5,5]],float)
after=bg+[12,-3];after[-1]+=[90,40]
shift,inliers,valid=estimate_camera_translation(bg,after)
assert valid and np.allclose(shift,[12,-3]) and inliers.tolist()==[True]*4+[False]
assert not estimate_camera_translation(bg[:2],after[:2])[2]
assert not estimate_camera_translation(np.empty((0,2)),np.empty((0,2)))[2]
for a,b in [(bg,after[:2]),(bg,np.full_like(bg,np.nan))]:
 try:estimate_camera_translation(a,b)
 except ValueError:pass
 else:raise AssertionError('invalid background arrays')
print('Camera translation: outlier, insufficient support, empty and invalid inputs passed')

## TODO 7 · Восстановимый журнал решения, а не только пары
Рассчитайте полную IoU-матрицу прогнозов и наблюдений. Сохраните исходные
track ID и detection indices, группы confidence, выбранные пары с указанием
стадии и две маски разрешённых связей. В high разрешены все живые tracks с
IoU ≥ gate. В low — только **ранее active** tracks, ещё не занятые high.
Колонки второй маски вне low и строки уже назначенных high должны быть False.

Журнал должен сериализоваться в JSON. Проверяйте индексы, взаимную однозначность,
gate и запрет использования lost в low; неправильная пара должна вызвать
ValueError. Такой audit позволяет доказать причину назначения и запрета.
Сравните IoU правильной и выбранной пары у пересечения; одного ID в логе мало.

In [ ]:
def association_audit(predicted, frame, track_ids, low_eligible, pairs, low, high, gate):
    """Return JSON-ready dict: track_ids, detection_indices, high, low, ignored,
    iou (NxM), high_allowed (NxM), low_allowed (NxM), pairs [[row,col,stage]].
    low_allowed removes HIGH-matched rows, requires low_eligible[row].
    Validate unique track IDs; one-to-one index pairs, confidence and IoU gate.
    A lost->low pair or any ineligible/duplicate pair raises ValueError."""
    raise NotImplementedError('TODO 7: association audit')

In [ ]:
pred=np.array([[0,0,10,10],[20,0,30,10]],float)
f={'boxes':pred.copy(),'scores':np.array([.9,.3])}
a=association_audit(pred,f,[8,9],[True,False],[(0,0)],.1,.6,.2)
assert a['high_allowed']==[[True,False],[False,False]] and not np.any(a['low_allowed'])
try:association_audit(pred,f,[8,9],[True,False],[(0,0),(1,1)],.1,.6,.2)
except ValueError:pass
else:raise AssertionError('lost track cannot use low')
a=association_audit(pred,f,[8,9],[True,True],[(0,0),(1,1)],.1,.6,.2)
assert a['pairs']==[[0,0,'high'],[1,1,'low']]
json.dumps(a)
print('Audit: stage masks, lost prohibition and serialization passed')

## TODO 8 · Полный переход трекера через кадр
Реализуйте `tracker_step`, а не только вызов matcher. Функция получает только
observations и внутреннее состояние. GT ей недоступна. Скопируйте входные tracks,
чтобы повторный запуск опыта не менял предыдущие состояния.

Порядок: оценка сдвига фона (если включена) → перенос box и last_box в текущую
систему координат → прогноз box + velocity → high → low только для ранее active →
update/reactivation либо miss/lost/removal → high births. Удаление: misses > max_age.
Реактивация lost возможна только через high; low продолжает ранее active, но
не восстанавливает старый lost. Абляция low_births меняет только births.

При update: gap=t−last_t; measured_velocity=(det[:2]−last_box[:2])/gap;
velocity=0.5·old_velocity+0.5·measured_velocity. После компенсации камера не должна
попадать в скорость объекта. Выдавайте только matched/new detections, потерянные
прогнозы сохраняйте внутри. В журнал включайте каждый match, birth, miss и remove.

Это учебная схема: без Kalman, unconfirmed и отдельного порога новых tracks.
Используется максимум числа допустимых пар, затем IoU; оригинальный ByteTrack
использует другой cost-limited assignment и дополнительные проверки.

In [ ]:
def tracker_step(tracks, next_id, frame, t, low, use_low=True,
                 low_births=False, compensate=False, max_age=2):
    """Return new_tracks,next_id,output,logs,audit; input tracks unchanged.
    Track fields: id,box[4],last_box[4],last_t,velocity[2],misses,state(active/lost).
    frame: boxes Nx4, scores N; optional background_before/background_after Nx2.
    Use HIGH,GATE, confidence_groups, two_stage(low_eligible=...),
    estimate_camera_translation, association_audit. Immediate high births.
    output={'ids':list,'boxes':Nx4}; logs have frame,track_id,det_index,score,event
    (miss/remove det_index=-1,score=0). audit adds frame,predicted,states_before,
    states_after,camera_shift,camera_valid. max_age nonnegative integer.
    Do not reuse a removed ID or emit lost forecasts as detected objects."""
    raise NotImplementedError('TODO 8: multi-frame tracking state transition')

In [ ]:
HIGH=.6;GATE=.2
f={'boxes':np.array([[0,0,10,10]],float),'scores':np.array([.9])}
empty={'boxes':np.empty((0,4)),'scores':np.empty(0)}
tracks,n,out,logs,a=tracker_step([],0,f,0,.1)
assert out['ids']==[0] and n==1
saved=tracks[0]['box'].copy()
lost,n,out,logs,a=tracker_step(tracks,n,empty,1,.1)
assert tracks[0]['state']=='active' and np.array_equal(tracks[0]['box'],saved)
assert lost[0]['state']=='lost' and not out['ids']
weak={**f,'scores':np.array([.3])}
still,n,out,logs,a=tracker_step(lost,n,weak,2,.1)
assert not out['ids'] and still[0]['misses']==2 # low cannot resurrect lost
back,n,out,logs,a=tracker_step(lost,n,f,2,.1)
assert out['ids']==[0] and logs[0]['event']=='reactivate high'
gone,n,out,logs,a=tracker_step(still,n,empty,3,.1)
assert gone==[] and logs[0]['event']=='remove'
new,n,out,logs,a=tracker_step(gone,n,f,4,.1)
assert out['ids']==[1] # ID never reused
bg=np.array([[0,0],[0,10],[10,0],[10,10]],float)
shifted={**f,'boxes':f['boxes']+[12,0,12,0],'background_before':bg,'background_after':bg+[12,0]}
cam,n,out,logs,a=tracker_step(tracks,1,shifted,1,.1,compensate=True)
assert out['ids']==[0] and np.allclose(cam[0]['velocity'],0) and a['camera_valid']
raw,n,out,logs,a=tracker_step(tracks,1,shifted,1,.1,compensate=False)
assert out['ids']==[1]
assert not tracker_step([],0,weak,0,.1)[2]['ids']
assert tracker_step([],0,weak,0,.1,low_births=True)[2]['ids']==[0]
print('Step integration: immutability, miss, low restriction, high reactivation, expiry, monotone IDs, camera/velocity and low birth ablation passed')

## TODO 9 · Собственные контрпримеры и ручной прогноз
Не менее трёх новых случаев: (1) приоритет high или запрет lost→low через несколько
кадров; (2) глобальное ID-соответствие либо association Jaccard; (3) несколько сдвигов
камеры с выбросами и проверкой скорости. Для каждого сначала запишите входы и
ожидаемые числа, затем assert и объяснение, какую ошибку реализации он обнаруживает.
Повтор открытых тестов с другими числами без новой ситуации не засчитывается.

In [ ]:
# TODO 9: at least THREE new hand-predicted examples with assertions.
# 1. High-first priority or the lost->low prohibition across frames.
# 2. Global ID assignment / HOTA association versus local TP counts.
# 3. Camera motion with outliers or repeated shifts without velocity leakage.
raise NotImplementedError('TODO 9: own mechanism and integration counterexamples')

## Сверка учебных счетчиков с TrackEval
Пять fixtures имеют однозначное matching и IoU только 0/1. Поэтому можно независимо проверить ваши счетчики и HOTA_alpha, не копируя сложный matching официальной метрики. В основной сцене ваш global IDF1 также сверяется с официальным Identity. Для crowded HOTA вызывается только официальная реализация.

In [ ]:
def identity_inputs(data):
 C=np.zeros((data['num_gt_ids'],data['num_tracker_ids']));g=np.zeros(data['num_gt_ids']);p=np.zeros(data['num_tracker_ids'])
 for gi,pi,sim in zip(data['gt_ids'],data['tracker_ids'],data['similarity_scores']):
  rr,cc=np.nonzero(sim>=.5);np.add.at(C,(gi[rr],pi[cc]),1);np.add.at(g,gi,1);np.add.at(p,pi,1)
 return C,g,p

def unambiguous_events(data):
 events=[]
 for g,p,sim in zip(data['gt_ids'],data['tracker_ids'],data['similarity_scores']):
  rr,cc=np.nonzero(sim>=.5)
  assert len(set(rr))==len(rr) and len(set(cc))==len(cc),'Fixture must have unambiguous matches'
  events.append({'gt_ids':g.tolist(),'pred_ids':p.tolist(),'matches':[(int(g[i]),int(p[j])) for i,j in zip(rr,cc)]})
 return events

fixtures={'perfect':simple_case([0]*4),'fragment':simple_case([0,0,1,1]),'gap_switch':simple_case([0,-1,1]),'all_missing':simple_case([-1]*4),'false_positives':simple_case([0]*4,2)}
fixture_rows=[]
for name,data in fixtures.items():
 full=official(data);row=metric_row(full);own=clear_counts(unambiguous_events(data));identity=global_id_scores(*identity_inputs(data))
 for our,their in [('TP','CLR_TP'),('FP','CLR_FP'),('FN','CLR_FN'),('IDSW','IDSW'),('MOTA','MOTA')]:assert np.isclose(own[our],row[their]),(name,our)
 for key,value in identity.items():assert np.isclose(value,row[key]),(name,key)
 # In these IoU 0/1 fixtures, compatible pairs are also unique matches at every alpha.
 C,g,p=identity_inputs(data);aggregation=hota_alpha_from_matches(C,g,p,int(row['CLR_FN']),int(row['CLR_FP']))
 for our,their in [('DetA','DetA'),('AssA','AssA'),('HOTA_alpha','HOTA')]:assert np.isclose(aggregation[our],row[their]),(name,our)
 fixture_rows.append({'case':name,**row})
print('Own counters agree with official TrackEval on five unambiguous fixtures.')

## Фиксированные observations и трекер
Точные GT boxes находятся отдельно от входа tracker. Scores снижаются в заданные интервалы, имитирующие потерю уверенности при перекрытии; само перекрытие не моделируется. Добавляются low и несколько high false candidates. Перемешивание порядка detections предотвращает утечку ID через индекс.

Трекер получает только observations. Его потерянные прогнозы остаются внутренними; вывод содержит только matched/new detections. max_age ограничивает память. Размер рамки копируется из последнего измерения, скорость обновляется по двум наблюдениям. Это прозрачная учебная динамика.

Ваш tracker_step вызывается на каждом кадре. Старый lost можно восстановить
только high, low продолжает ранее active. Камера в основном val/test неподвижна;
её компенсация проверяется отдельно в stress. В audit нет GT: только прогнозы,
наблюдения, состояния и решения. Индексы detections локальны для кадра.

In [ ]:
def make_sequence(seed,T=40):
 rng=np.random.default_rng(seed);gt=[];observed=[]
 for t in range(T):
  centers=np.array([[20+1.4*t,30.],[100-1.4*t,35.]])
  true=np.c_[centers[:,0]-4,centers[:,1]-6,centers[:,0]+4,centers[:,1]+6]
  gt.append({'ids':[0,1],'boxes':true})
  boxes=[];scores=[]
  for i in range(2):
   if rng.random()<.04:continue
   jitter=rng.normal(0,.6,2);box=true[i]+np.r_[jitter,jitter]
   score=rng.uniform(.24,.52) if ((i==0 and 12<=t<=18) or (i==1 and 25<=t<=29)) else rng.uniform(.72,.98)
   boxes.append(box);scores.append(score)
  # Low-confidence false candidates never perfectly explain their source.
  if t%4==0:
   cx,cy=rng.uniform([10,15],[105,55]);boxes.append([cx-4,cy-6,cx+4,cy+6]);scores.append(rng.uniform(.12,.38))
  if t in [9,30]:
   cx,cy=rng.uniform([10,15],[105,55]);boxes.append([cx-4,cy-6,cx+4,cy+6]);scores.append(.8)
  order=rng.permutation(len(scores));observed.append({'boxes':np.array(boxes,float).reshape(-1,4)[order],'scores':np.array(scores)[order]})
 return gt,observed

VAL_SEEDS=[101,102,103,104];TEST_SEEDS=[201,202,203,204]
assert set(VAL_SEEDS).isdisjoint(TEST_SEEDS)
val_sequences={seed:make_sequence(seed) for seed in VAL_SEEDS}
HIGH=.6;GATE=.2;MAX_AGE=2

In [ ]:
LAST_AUDITS=[]
def run_tracker(observations,low,use_low=True,low_births=False,compensate=False,max_age=None):
 global LAST_AUDITS
 tracks=[];next_id=0;output=[];logs=[];LAST_AUDITS=[];start=time.perf_counter()
 for t,frame in enumerate(observations):
  tracks,next_id,out,events,audit=tracker_step(tracks,next_id,frame,t,low,use_low,low_births,compensate,MAX_AGE if max_age is None else max_age)
  output.append(out);logs.extend(events);LAST_AUDITS.append(audit)
 return output,logs,time.perf_counter()-start

def evaluate(gt,observed,low,use_low=True,low_births=False,compensate=False,max_age=None):
 pred,logs,seconds=run_tracker(observed,low,use_low,low_births,compensate,max_age)
 data=make_metric_data(gt,pred);results=official(data)
 ours=global_id_scores(*identity_inputs(data))
 for key,value in ours.items():assert np.isclose(value,results['identity'][key]),key
 return {**metric_row(results),'tracker_seconds':seconds},results,pred,logs

def combine(sequence_results):
 return {name:metric.combine_sequences({str(i):r[name] for i,r in enumerate(sequence_results)}) for name,metric in METRICS.items()}

## Val: выбрать low до test
Проверьте каждую sequence отдельно, затем официальное объединение. Объясните компромисс FN/FP/IDSW. Не заменяйте HOTA критерий после просмотра результата.

In [ ]:
RESULTS=Path('results');RESULTS.mkdir(exist_ok=True)
def write_csv(name,rows):
 with (RESULTS/name).open('w',newline='') as f:
  writer=csv.DictWriter(f,fieldnames=rows[0]);writer.writeheader();writer.writerows(rows)
write_csv('metric_fixture_checks.csv',fixture_rows)
LOWS=[.1,.25,.4];val_rows=[];val_summary=[]
for low in LOWS:
 all_results=[]
 for seed,(gt,observed) in val_sequences.items():
  row,results,_,_=evaluate(gt,observed,low);val_rows.append({'seed':seed,'low':low,**row});all_results.append(results)
 val_summary.append({'low':low,**metric_row(combine(all_results))})
# Official sequence pooling before averaging HOTA's alpha values.
selected=max(val_summary,key=lambda r:(r['HOTA'],r['low']))
lock={'low':selected['low'],'high':HIGH,'gate':GATE,'max_age':MAX_AGE,'selection':'max official pooled val HOTA; ties larger low','val_seeds':VAL_SEEDS,'test_seeds':TEST_SEEDS,'trackeval_revision':TRACKEVAL_REV}
(RESULTS/'selection_locked.json').write_text(json.dumps(lock,indent=2))
write_csv('validation.csv',val_rows);write_csv('validation_summary.csv',val_summary)
for row in val_summary:print(row)
print('LOCKED before creating test:',lock)
fig,ax=plt.subplots(figsize=(8,3))
for key in ['MOTA','IDF1','HOTA']:ax.plot(LOWS,[r[key] for r in val_summary],'o-',label=key)
ax.set(xlabel='low confidence threshold',ylabel='official pooled score',ylim=(0,1));ax.legend();ax.grid();fig.tight_layout();fig.savefig(RESULTS/'validation.png');plt.show();plt.close(fig)

## Решение перед test
**Заполните:** выбранный low, pooled val MOTA/IDF1/HOTA, FP/FN/IDSW, ожидаемый эффект запрета low births. Сохраненный `selection_locked.json` задает параметры. Далее открывается общий фиксированный test всех трех заранее объявленных вариантов.

In [ ]:
lock=json.loads((RESULTS/'selection_locked.json').read_text());test_rows=[];pooled={};artifacts={};all_logs=[]
# Predeclared comparisons: second-stage benefit and low-birth ablation.
for name,use_low,low_births in [('high_only',False,False),('two_stage',True,False),('low_birth_ablation',True,True)]:
 results_by_seed=[]
 for seed in TEST_SEEDS:
  gt,observed=make_sequence(seed);row,results,pred,logs=evaluate(gt,observed,lock['low'],use_low,low_births)
  test_rows.append({'method':name,'seed':seed,**row});results_by_seed.append(results);artifacts[(name,seed)]=(gt,pred)
  all_logs.extend({'method':name,'seed':seed,**r} for r in logs)
 pooled[name]=combine(results_by_seed)
summary=[{'method':name,**metric_row(res)} for name,res in pooled.items()]
write_csv('test.csv',test_rows);write_csv('test_summary.csv',summary);write_csv('association_log.csv',all_logs)
for row in summary:print(row)
fig,axs=plt.subplots(1,2,figsize=(11,3.5),constrained_layout=True)
for key in ['MOTA','IDF1','HOTA']:axs[0].plot(range(3),[r[key] for r in summary],'o-',label=key)
axs[0].set(xticks=range(3),xticklabels=['high only','two stage','low births'],ylim=(0,1),ylabel='official pooled score');axs[0].legend();axs[0].grid()
for name,res in pooled.items():axs[1].plot(METRICS['hota'].array_labels,res['hota']['HOTA'],label=name)
axs[1].set(xlabel='alpha',ylabel='HOTA at alpha',ylim=(0,1));axs[1].legend();axs[1].grid();fig.savefig(RESULTS/'test_metrics.png');plt.show();plt.close(fig)
# Same first test scene, same frames, all boxes for each method.
fig,axs=plt.subplots(3,3,figsize=(11,7),constrained_layout=True)
for row,name in enumerate(pooled):
 gt,pred=artifacts[(name,TEST_SEEDS[0])]
 for col,t in enumerate([11,15,19]):
  ax=axs[row,col]
  for b in gt[t]['boxes']:ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color='black',ls='--',lw=1.5))
  for tid,b in zip(pred[t]['ids'],pred[t]['boxes']):
   color=plt.cm.tab10(tid%10);ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color=color,lw=2));ax.text(b[0],b[1]-1,str(tid),color=color)
  ax.set(xlim=(5,115),ylim=(58,8),title=f'{name}, frame {t}',aspect='equal')
fig.savefig(RESULTS/'same_frames.png');plt.show();plt.close(fig)

## Контроль причин: три заранее заданных stress только на validation
Выбранный low не меняется. Для каждого val seed создайте пары «исходные данные →
изменение одной причины»: (1) добавлены только low-кандидаты; (2) удалены все
observations кадров 14–18; (3) в кадре 20 камера сдвинулась на 18 пикселей,
и доступны фоновые соответствия с выбросами. GT камера также переносит, но
трекер получает только observations и фоновые пары.

Сравнения объявлены заранее: high-only/two-stage/low-births в low-clutter;
max_age=2/6 при длинном пропуске; GMC off/on при camera jump. Везде сохраняются
остальные параметры и одинаковые данные. Метрики официально объединяются по
sequences. Это объясняющие опыты после выбора, а не новый подбор по test.

In [ ]:
import copy
def stress_sequence(seed,scenario):
 gt,observed=make_sequence(seed);gt=copy.deepcopy(gt);observed=copy.deepcopy(observed)
 rng=np.random.default_rng(seed+9000)
 bg=np.array([[x,y] for x in [0,30,60] for y in [0,30,60]],float)
 for t,frame in enumerate(observed):
  if scenario=='low_clutter' and len(frame['boxes']):
   clutter=frame['boxes']+np.array([3.,0,3.,0]);frame['boxes']=np.concatenate([frame['boxes'],clutter]);frame['scores']=np.r_[frame['scores'],np.full(len(clutter),.28)]
  elif scenario=='long_gap' and 14<=t<=18:frame.update(boxes=np.empty((0,4)),scores=np.empty(0))
  elif scenario=='camera_jump':
   shift=np.array([18.,0]) if t>=20 else np.zeros(2)
   gt[t]['boxes']=gt[t]['boxes']+np.tile(shift,2);frame['boxes']=frame['boxes']+np.tile(shift,2)
   delta=np.array([18.,0]) if t==20 else np.zeros(2)
   after=bg+delta+rng.normal(0,.05,bg.shape);after[-2:]+=[25,-30]
   frame.update(background_before=bg.copy(),background_after=after)
 return gt,observed

locked_bytes=(RESULTS/'selection_locked.json').read_bytes();stress_rows=[];stress_summary=[];audits=[];stress_artifacts={}
plans={'low_clutter':[('high_only',{'use_low':False}),('two_stage',{}),('low_births',{'low_births':True})],
       'long_gap':[('age2',{'max_age':2}),('age6',{'max_age':6})],
       'camera_jump':[('raw',{}),('compensated',{'compensate':True})]}
for scenario,variants in plans.items():
 for method,options in variants:
  sequence_results=[]
  for seed in VAL_SEEDS:
   gt,observed=stress_sequence(seed,scenario);row,res,pred,logs=evaluate(gt,observed,lock['low'],**options)
   stress_rows.append({'scenario':scenario,'method':method,'seed':seed,**row});sequence_results.append(res)
   if seed==VAL_SEEDS[0]:
    stress_artifacts[(scenario,method)]=(gt,pred)
    audits.extend({'scenario':scenario,'method':method,'seed':seed,**a} for a in LAST_AUDITS if a['frame'] in [13,17,19,20,21,25])
  stress_summary.append({'scenario':scenario,'method':method,**metric_row(combine(sequence_results))})
assert (RESULTS/'selection_locked.json').read_bytes()==locked_bytes
# A named diagnostic fixture guarantees that each lifecycle transition can be
# inspected, regardless of random val scenes. It is not pooled with val metrics.
box=np.array([[0,0,10,10]],float)
fixture=[{'boxes':box.copy() if score is not None else np.empty((0,4)),
          'scores':np.array([score]) if score is not None else np.empty(0)}
         for score in [.9,None,.3,.9,None,None,None,.9]]
_,fixture_events,_=run_tracker(fixture,.1)
audits.extend({'scenario':'lifecycle_fixture','method':'two_stage','seed':None,
               **a,'events':[e for e in fixture_events if e['frame']==a['frame']]} for a in LAST_AUDITS)
write_csv('stress_validation.csv',stress_rows);write_csv('stress_summary.csv',stress_summary)
(RESULTS/'association_audit.json').write_text(json.dumps(audits,indent=2))
for row in stress_summary:print(row)
fig,axs=plt.subplots(1,3,figsize=(13,3.5),constrained_layout=True)
for ax,scenario in zip(axs,plans):
 rows=[r for r in stress_summary if r['scenario']==scenario]
 for metric in ['MOTA','IDF1','HOTA']:ax.plot(range(len(rows)),[r[metric] for r in rows],'o-',label=metric)
 lower=min(0.,min(r['MOTA'] for r in rows)-.1)
 ax.set(xticks=range(len(rows)),xticklabels=[r['method'] for r in rows],title=scenario,ylim=(lower,1.05));ax.grid();ax.legend(fontsize=8)
fig.savefig(RESULTS/'stress_validation.png');plt.show();plt.close(fig)
fig,axs=plt.subplots(2,3,figsize=(12,6),constrained_layout=True)
for col,(scenario,methods,t) in enumerate([('low_clutter',['two_stage','low_births'],16),('long_gap',['age2','age6'],19),('camera_jump',['raw','compensated'],20)]):
 for row,method in enumerate(methods):
  gt,pred=stress_artifacts[(scenario,method)];ax=axs[row,col]
  for b in gt[t]['boxes']:ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color='black',ls='--'))
  ordered=sorted(zip(pred[t]['ids'],pred[t]['boxes']),key=lambda item:item[1][0])
  for k,(tid,b) in enumerate(ordered):
   color=plt.cm.tab10(tid%10);ax.add_patch(Rectangle(b[:2],*(b[2:]-b[:2]),fill=False,color=color,lw=2))
   if scenario=='low_clutter' and method=='low_births':
    ax.annotate(f'T{tid}',xy=(b[:2]+b[2:])/2,xytext=(5+130*(k+.5)/len(ordered),5+8*(k%2)),fontsize=8,color=color,ha='center',arrowprops={'arrowstyle':'-','color':color,'lw':.6})
   else:ax.text(b[0],b[1]-1,str(tid),color=color)
  ax.set(xlim=(0,140),ylim=(65,0),aspect='equal',title=f'{scenario}: {method}, frame {t}')
fig.savefig(RESULTS/'stress_same_frames.png');plt.show();plt.close(fig)

### Разбор stress-опытов

В `association_audit.json` восстановите решения на одном low-кадре, на
пересечении, при удалении и реактивации, до/после компенсации камеры. Для
каждого случая выпишите строку матрицы, разрешённость и конкретный ID.
Если механизм ухудшил результат, найдите событие, объясняющее ухудшение.

## Реальный клип: компоненты → две стадии → траектории
MOG2 обрабатывает первые 80 кадров OpenCV vtest при размере 384×288; первые
20 кадров прогревают фон и не поступают в трекер. После этого **один и тот же
список рамок** подаётся в high-only и двухстадийный вариант. Здесь score =
min(area / 300, 1) — искусственная оценка площади foreground-компоненты,
**не confidence детектора людей**. Заранее фиксированы low = .1 и high = .6:
low соответствует площади от 30 до 180 пикселей, high — от 180 пикселей.
Выбор low на синтетике на этот proxy не переносится. GT отсутствует: MOTA,
IDF1, HOTA, FP/FN и IDSW здесь не вычисляются.


На рисунке верхняя подпись — исходный номер кадра; все CSV сохраняют его
отдельно от индекса шага трекера. Исходный ролик скачивается в временный кэш
с проверкой SHA-256; можно задать `COURSE_VIDEO_PATH` к той же копии. Само
видео и его кадры в общий репозиторий не включайте.

In [ ]:
import cv2, urllib.request, tempfile
VIDEO_URL = 'https://raw.githubusercontent.com/opencv/opencv/4.10.0/samples/data/vtest.avi'
VIDEO_SHA = '45cddc9490be69345cbdab64ca583be65987e864ca408038e648db99e10516cf'
video_cache = Path(tempfile.gettempdir()) / 'applied-ai-tracking'
video_cache.mkdir(exist_ok=True)
video_path = Path(os.environ.get('COURSE_VIDEO_PATH', video_cache / 'vtest.avi'))
if not video_path.exists():
    with urllib.request.urlopen(VIDEO_URL, timeout=60) as response:
        video_path.write_bytes(response.read())
if hashlib.sha256(video_path.read_bytes()).hexdigest() != VIDEO_SHA:
    raise ValueError('Видео не совпадает с проверенным OpenCV vtest.avi (SHA-256).')

VIDEO_WARMUP = 20
VIDEO_FRAMES = 80
VIDEO_LOW = .1
assert HIGH == .6, 'Real-video area protocol requires high=.6.'
cap = cv2.VideoCapture(str(video_path))
if not cap.isOpened():
    raise RuntimeError('OpenCV не открыл проверенный файл vtest.avi.')
mog = cv2.createBackgroundSubtractorMOG2(history=100, varThreshold=25, detectShadows=True)
video_observed, video_components, video_images, video_masks = [], [], {}, {}
video_started = time.perf_counter()
try:
    for source_frame in range(VIDEO_FRAMES):
        ok, bgr = cap.read()
        if not ok:
            raise RuntimeError(f'Видео закончилось до кадра {source_frame}.')
        bgr = cv2.resize(bgr, (384, 288))
        mask = np.uint8(mog.apply(bgr) == 255) * 255
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
        if source_frame < VIDEO_WARMUP:
            continue
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Deterministic indices D refer to this frame's accepted component list.
        components = sorted([(cv2.boundingRect(c), float(cv2.contourArea(c)))
                             for c in contours if cv2.contourArea(c) >= 30.],
                            key=lambda item: (item[0][0], item[0][1], item[0][2], item[0][3]))
        boxes, scores = [], []
        for det_index, ((x, y, w, h), area) in enumerate(components):
            score = min(area / 300., 1.)
            boxes.append([x, y, x + w, y + h]); scores.append(score)
            video_components.append({'source_frame': source_frame,
                                     'frame': source_frame - VIDEO_WARMUP,
                                     'det_index': det_index, 'area': area,
                                     'proxy_score': score,
                                     'x1': x, 'y1': y, 'x2': x + w, 'y2': y + h})
        video_observed.append({'boxes': np.asarray(boxes, float).reshape(-1, 4),
                               'scores': np.asarray(scores, float)})
        video_images[source_frame] = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        video_masks[source_frame] = mask.copy()
finally:
    cap.release()
video_preprocess_seconds = time.perf_counter() - video_started

video_predictions, video_logs, video_track_rows, video_runtime = {}, [], [], {}
video_audits=[]
for method, use_low in [('high_only', False), ('two_stage', True)]:
    pred, logs, seconds = run_tracker(video_observed, VIDEO_LOW, use_low=use_low,
                                      low_births=False, compensate=False)
    video_predictions[method] = pred
    video_runtime[method] = seconds
    video_audits.extend({'method':method,**a,'source_frame':int(a['frame'])+VIDEO_WARMUP}
                        for a in LAST_AUDITS)
    for row in logs:
        video_logs.append({'method': method, **row,
                           'source_frame': int(row['frame']) + VIDEO_WARMUP})
    for frame_index, frame_pred in enumerate(pred):
        for tid, box in zip(frame_pred['ids'], frame_pred['boxes']):
            video_track_rows.append({'method': method, 'frame': frame_index,
                                     'source_frame': frame_index + VIDEO_WARMUP,
                                     'track_id': int(tid),
                                     **{key: float(value) for key, value in
                                        zip(['x1', 'y1', 'x2', 'y2'], box)}})

def video_csv(name, rows, fallback_fields):
    fields = list(dict.fromkeys(key for row in rows for key in row)) or fallback_fields
    with (RESULTS / name).open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader(); writer.writerows(rows)

video_csv('real_video_components.csv', video_components,
          ['source_frame', 'frame', 'det_index', 'area', 'proxy_score', 'x1', 'y1', 'x2', 'y2'])
video_csv('real_video_tracks.csv', video_track_rows,
          ['method', 'frame', 'source_frame', 'track_id', 'x1', 'y1', 'x2', 'y2'])
video_csv('real_video_association.csv', video_logs,
          ['method', 'frame', 'source_frame', 'track_id', 'det_index', 'event'])
(RESULTS/'real_video_audit.json').write_text(json.dumps(video_audits,indent=2))

fig, axes = plt.subplots(3, 3, figsize=(15, 11.6), constrained_layout=True)
for row_index, source_frame in enumerate([30, 50, 79]):
    frame_index = source_frame - VIDEO_WARMUP
    obs = video_observed[frame_index]
    for col_index in range(3):
        ax = axes[row_index, col_index]
        ax.imshow(video_images[source_frame]); ax.set_xlim(0, 384); ax.set_ylim(288, 0)
        ax.set_xticks([]); ax.set_yticks([])
        if col_index == 0:
            for j, (box, score) in enumerate(zip(obs['boxes'], obs['scores'])):
                colour = 'lime' if score >= HIGH else 'orange'
                ax.add_patch(Rectangle(box[:2], *(box[2:] - box[:2]), fill=False,
                                       color=colour, lw=1.5))
                ax.text(box[0], max(10., box[1] - 3), f'D{j} {score:.2f}',
                        fontsize=7, color='white',
                        bbox={'facecolor': 'black', 'alpha': .65, 'pad': .8})
            ax.set_title(f'Frame {source_frame}: same components (D)')
            inset = ax.inset_axes([.70, .01, .29, .27])
            inset.imshow(video_masks[source_frame], cmap='gray', vmin=0, vmax=255)
            inset.set_xticks([]); inset.set_yticks([]); inset.set_title('MOG2 mask', fontsize=7)
        else:
            method = ['high_only', 'two_stage'][col_index - 1]
            frame_pred = video_predictions[method][frame_index]
            for tid, box in zip(frame_pred['ids'], frame_pred['boxes']):
                colour = plt.cm.tab10(int(tid) % 10)
                ax.add_patch(Rectangle(box[:2], *(box[2:] - box[:2]), fill=False,
                                       color=colour, lw=2))
                trail = []
                for past in video_predictions[method][max(0, frame_index - 10):frame_index + 1]:
                    if tid in past['ids']:
                        b = np.asarray(past['boxes'][list(past['ids']).index(tid)])
                        trail.append((b[:2] + b[2:]) / 2)
                if len(trail) > 1:
                    trail = np.asarray(trail); ax.plot(trail[:, 0], trail[:, 1], '-',
                                                      color=colour, lw=1.3, alpha=.8)
                associated = [r for r in video_logs if r['method'] == method
                              and r['frame'] == frame_index and r.get('track_id') == tid
                              and isinstance(r.get('det_index'), (int, np.integer))
                              and r['det_index'] >= 0]
                label = f'T{tid}'
                if associated:
                    label += f" / D{associated[-1]['det_index']}"
                ax.text(box[0], max(10., box[1] - 3), label, fontsize=8, color='white',
                        bbox={'facecolor': 'black', 'alpha': .65, 'pad': .8})
            ax.set_title(f"Frame {source_frame}: {method}, IDs={frame_pred['ids']}", fontsize=9)
fig.savefig(RESULTS / 'real_video_tracking.png', dpi=140)
plt.show(); plt.close(fig)

video_report = {'sha256': VIDEO_SHA, 'frames_decoded': VIDEO_FRAMES,
                'warmup_frames': VIDEO_WARMUP, 'frames_tracked': len(video_observed),
                'source_frame_origin': 0, 'display_frames': [30, 50, 79],
                'resize': [384, 288], 'low': VIDEO_LOW, 'high': HIGH,
                'score': 'min(foreground component contour area / 300, 1); NOT detector confidence',
                'minimum_component_area': 30, 'ground_truth': False,
                'preprocess_seconds': video_preprocess_seconds,
                'tracker_seconds': video_runtime, 'opencv': cv2.__version__,
                'metrics': 'No GT metrics, FP/FN or IDSW computed',
                'same_observations': True,
                'limitations': ['MOG2 components can merge people, split one person or describe background',
                                'area proxy is not a calibrated score and thresholds are fixed without selection',
                                'colour is predicted track ID, not verified human identity']}
(RESULTS / 'real_video_run.json').write_text(json.dumps(video_report, indent=2))
for source_frame in [30, 50, 79]:
    print('Original frame', source_frame)
    for method in video_predictions:
        print(method, 'issued IDs:', video_predictions[method][source_frame - VIDEO_WARMUP]['ids'])
        print('events:', [r for r in video_logs if r['method'] == method
                          and r['source_frame'] == source_frame])
print('Same MOG2 components were tracked twice. No GT metrics were computed.')

### Проверка на реальном клипе

До запуска предскажите, когда маленькая компонента сможет продолжить ранее
созданную траекторию, а когда её нельзя связать с человеком. После запуска:

1. На кадрах 30, 50 и 79 сопоставьте **тот же component D и track T** у двух
   вариантов. По таблице событий найдите high match, low match и рождение;
   если одного события на этих кадрах нет, выберите другой кадр из CSV и
   укажите его номер. Отсутствие полезного low match тоже допустимый результат.
2. Найдите одну устойчивую видимую траекторию и один подозрительный переход
   или разрыв. Покажите рамки до/после, score, IoU и событие; объясните, могло
   ли причиной быть дробление или слияние foreground-компонент.
3. Вырежьте из журнала путь одного track через несколько кадров. Чем отличается
   внутреннее состояние lost от выданной рамки? Почему число выданных ID
   без GT не позволяет назвать один метод качественнее другого?

## Сдача и выводы
1. Девять TODO, три собственных контрпримера с прогнозами, официальная сверка,
   таблица минимум **18 опытов демо** — по два на каждый стенд.
2. `metric_fixture_checks.csv`, val/test CSV, `selection_locked.json`,
   `association_log.csv`, `association_audit.json`, stress CSV и графики.
3. Восстановите по матрице один полезный low match, один запрет lost→low,
   high-реактивацию и удаление. Укажите seed/frame, track ID, detection index,
   IoU, границы confidence и состояния до/после. Для реальной ошибки в пересечении
   сопоставьте тот же GT и кадры у baseline и two-stage.
4. Сравните stress по одной причине: фоновые low, длинный пропуск, камера.
   Покажите, какие FP/FN/IDSW объясняют изменение DetA/AssA/HOTA.
   Разберите случай, где полезное снижение FN не спасло identity.
5. Видеочасть: три наблюдения, путь одного ID, исходные номера кадров и
   `real_video_components.csv`, `real_video_tracks.csv`, `real_video_association.csv`, `real_video_audit.json`,
   `real_video_run.json`, `real_video_tracking.png`. GT отсутствует.

Выбор low фиксируется до test, stress выполняется только на val без перенастройки.
Ухудшение при корректном протоколе допустимо. В отчёте отличите учебные механизмы
от ByteTrack/BoT-SORT, цветовой признак от обученного re-ID и proxy площади от
confidence детектора. Укажите ограничения синтетики и происхождение данных.

Источники: [ByteTrack](https://arxiv.org/abs/2110.06864),
[код ByteTrack](https://github.com/FoundationVision/ByteTrack/blob/main/yolox/tracker/byte_tracker.py),
[BoT-SORT](https://arxiv.org/abs/2206.14651), [HOTA](https://arxiv.org/abs/2009.07736),
[TrackEval](https://github.com/JonathonLuiten/TrackEval/tree/12c8791b303e0a0b50f753af204249e622d0281a).